# Solar Filament v4 — Edge-guided Attention U-Net (research candidate)

Train a **new** edge-guided bottleneck-attention model from scratch on official MAGFiLO training images. The architecture is inspired by the published [EdgeAttNet paper](https://arxiv.org/abs/2509.02964), **not** an exact reproduction. A soft-clDice loss is available as a separate ablation based on [CVPR 2021](https://openaccess.thecvf.com/content/CVPR2021/html/Shit_clDice_-_A_Novel_Topology-Preserving_Loss_Function_for_Tubular_Structure_CVPR_2021_paper.html).

**Requirements:** Kaggle GPU + Internet On + official competition input. No previous notebook output is needed. One Kaggle T4 is used by standard PyTorch even when T4 x2 is selected. **Run this only after finishing the cheaper v3 experiment**; do not run both notebooks at once.

A better public score is **not guaranteed**. The notebook runs CPU synthetic regression tests, date-grouped training, PQ/instance-Dice proxy tuning with a disjoint confirmation subset, and 180-image inference. Save the output checkpoint and do not use any externally leaked test labels.

In [ ]:
import sys, subprocess, pathlib, torch
ROOT=pathlib.Path('/kaggle/working/Solar-Filament-Segmentation')
assert torch.cuda.is_available(), 'Enable GPU in Kaggle notebook options.'
assert not ROOT.exists(), 'Restart in a fresh Kaggle session.'
subprocess.run(['git','clone','--depth','1','https://github.com/ahpa5246-lgtm/Solar-Filament-Segmentation.git',str(ROOT)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','pycocotools==2.0.8'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(ROOT)],check=True)
sys.path.insert(0,str(ROOT))
from solar.data import discover
print('GPU:',torch.cuda.get_device_name(0))
print('Training annotations:',discover('/kaggle/input')[0])
subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests')],check=True)

## 1 — Architecture and training
Default `CLDICE_WEIGHT=0` isolates the **architecture** change. A separate run with `0.05` would test the topology loss; do not attribute joint changes to one component.

In [ ]:
BATCH=2
EPOCHS=30
STEPS=160
CLDICE_WEIGHT=0.0
OUT='/kaggle/working/v4_artifacts'
subprocess.run([sys.executable,'-m','solar.train','--data','/kaggle/input',
                '--out',OUT,'--epochs',str(EPOCHS),'--steps-per-epoch',str(STEPS),
                '--batch',str(BATCH),'--seed','42','--architecture','edge_attention',
                '--cldice-weight',str(CLDICE_WEIGHT),'--val-patches','64'],check=True)
assert pathlib.Path(OUT,'best.pt').exists(), 'Training must finish and produce a trained checkpoint.'
print('Trained:',OUT+'/best.pt')

## 2 — Morphology-aware calibration
Use the same TTA in tuning and inference. Report independent confirmation and compare with the baseline threshold; never use the Kaggle test labels.

In [ ]:
TTA='flip'
subprocess.run([sys.executable,'-m','solar.tune','--data','/kaggle/input',
                '--checkpoint',OUT+'/best.pt','--out',OUT,'--profile','precision',
                '--max-images','24','--audit-images','12','--tta-mode',TTA,
                '--batch',str(BATCH),'--seed','42'],check=True)
import json
report=json.load(open(OUT+'/confirmation_results.json'))
print('Confirmation:',json.dumps(report['results'],indent=2))

## 3 — Predict every official test image
No debug limit. The resulting submission is a NEW candidate and must be evaluated by Kaggle before claiming any improvement.

In [ ]:
subprocess.run([sys.executable,'-m','solar.predict','--data','/kaggle/input',
                '--checkpoint',OUT+'/best.pt','--params',OUT+'/best_params.json',
                '--out','/kaggle/working/submission.csv','--batch',str(BATCH),
                '--tta-mode',TTA],check=True)
import pandas as pd
s=pd.read_csv('/kaggle/working/submission.csv',dtype=str)
assert list(s.columns)==['filament_id','segmentation_rle']
assert s.filament_id.is_unique and s.segmentation_rle.notna().all()
print('Predicted filaments:',len(s),'Images with detections:',s.filament_id.str.rsplit('_',n=1).str[0].nunique())

## 4 — Reproducibility
Save this notebook version and all artifacts. Compare official v4 score with **0.23**, not with the held-out PQ proxy.

In [ ]:
subprocess.run([sys.executable,'-m','pip','freeze'],stdout=open(OUT+'/environment_frozen.txt','w'),check=True)
print('Model and validation evidence:',list(pathlib.Path(OUT).iterdir()))
print('CSV ready at /kaggle/working/submission.csv')